# 🎵 Proyecto ML — Spotify: Clasificación Multiclase de Popularidad

## Entrega Final — Pipeline End-to-End y Comparación de 3 Técnicas

**Cambio de arquitectura respecto a Fase 2:** el problema pasa de **regresión continua** a **clasificación multiclase** sobre rangos de popularidad.

**Modelos comparados:**
1. `DecisionTreeClassifier` — modelo base interpretable
2. `RandomForestClassifier` — ensemble por bagging
3. `SVC` (kernel RBF) — método de márgenes sobre features escaladas

**Decisiones de ingeniería críticas (justificadas en cada sección):**
- Binning del target con **terciles dinámicos** para evitar desbalance severo
- **Split estratificado ANTES** de cualquier encoding derivado del target → cero data leakage
- `TargetMeanEncoder` custom compatible con API de sklearn
- SVC sobre **subsample estratificado** para que el entrenamiento sea factible (justificado en §6.3)

---

## 0. Setup

Imports agrupados y semilla global. Todo el pipeline es reproducible — misma semilla, mismo resultado.

In [ ]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass, field
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Entorno listo')

In [ ]:
!pip install kagglehub -q

In [ ]:
import kagglehub

path = kagglehub.dataset_download('maharshipandya/-spotify-tracks-dataset')
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
DATASET_PATH = os.path.join(path, csv_files[0])
print(f'✅ Dataset en: {DATASET_PATH}')

---
## 1. Carga y Limpieza Inicial

**Decisiones de esta sección:**
1. **Columnas descartadas:** `track_id`, `track_name`, `artists`, `album_name` — son identificadores/strings de alta cardinalidad sin señal generalizable. Mantenerlos solo aumenta el uso de memoria.
2. **Duplicados:** eliminamos tanto duplicados exactos como duplicados por `track_id` (una misma canción puede aparecer en varios álbumes/géneros). Mantenerlos infla artificialmente el peso de esas observaciones y favorece el overfitting.
3. **`explicit` a `int`:** los modelos esperan tipos numéricos.

In [ ]:
def load_and_clean(path: str) -> pd.DataFrame:
    """Carga el CSV crudo y aplica limpieza determinista mínima.

    Operaciones:
        - Eliminación de columnas identificadoras (sin valor predictivo).
        - Drop de filas con nulos (~irrelevante para 114k filas).
        - Drop de duplicados exactos y por ``track_id``.
        - Cast de ``explicit`` a entero.

    Args:
        path: ruta absoluta al CSV del dataset.

    Returns:
        DataFrame limpio listo para feature engineering.
    """
    raw = pd.read_csv(path, index_col=0)
    n0 = len(raw)

    # Duplicados por track_id ANTES de drop de columnas (necesitamos la columna)
    if 'track_id' in raw.columns:
        raw = raw.drop_duplicates(subset=['track_id'], keep='first')

    drop_cols = ['track_id', 'track_name', 'artists', 'album_name']
    raw = raw.drop(columns=[c for c in drop_cols if c in raw.columns])

    raw = raw.dropna().drop_duplicates().reset_index(drop=True)

    if 'explicit' in raw.columns:
        raw['explicit'] = raw['explicit'].astype(int)

    print(f'Filas iniciales: {n0:,}  →  finales: {len(raw):,}  ({n0 - len(raw):,} descartadas)')
    return raw

df = load_and_clean(DATASET_PATH)
df.head()

---
## 2. Definición del Target Multiclase

### 2.1 Diagnóstico: ¿por qué NO usar cortes fijos [0-30], [31-70], [71-100]?

La distribución de `popularity` está fuertemente sesgada a la izquierda. Si usamos esos cortes, la clase `Alta` puede tener <2% del dataset. Un clasificador que **siempre prediga `Baja`** tendría accuracy ~90% pero F1 macro miserable — métrica engañosa.

### 2.2 Solución: terciles dinámicos (`pd.qcut`)

Usamos los terciles **empíricos** (q=3) para garantizar ~33% por clase. Los cortes reales se loguean y se usan como justificación en la exposición.

> **Trade-off explícito:** perdemos interpretabilidad semántica ("≥70 es viral") a cambio de un problema de clasificación balanceado. Para el alcance escolar y la honestidad estadística, gana la segunda.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['popularity'], bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_title('Distribución original de popularity')
axes[0].set_xlabel('popularity (0-100)')

# Simulación: cómo quedaría con cortes fijos [0-30], [31-70], [71-100]
fixed_bins = pd.cut(df['popularity'], bins=[-1, 30, 70, 100], labels=['Baja', 'Media', 'Alta'])
fixed_counts = fixed_bins.value_counts().sort_index()
axes[1].bar(fixed_counts.index.astype(str), fixed_counts.values, color=['#5B8DB8', '#E89B3C', '#C0392B'])
axes[1].set_title('Si usáramos cortes fijos → desbalance severo')
for i, v in enumerate(fixed_counts.values):
    axes[1].text(i, v, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[1].set_ylim(0, fixed_counts.max() * 1.2)
plt.tight_layout(); plt.show()

In [ ]:
POP_NUMERIC_COL = 'popularity'   # columna original (se conserva para target encoding)
POP_CLASS_COL   = 'popularity_class'
CLASS_LABELS    = ['Baja', 'Media', 'Alta']

df[POP_CLASS_COL], bin_edges = pd.qcut(
    df[POP_NUMERIC_COL],
    q=3,
    labels=CLASS_LABELS,
    retbins=True,
    duplicates='drop',
)

print('Cortes (bin edges) reales calculados:')
for i, label in enumerate(CLASS_LABELS):
    print(f'  {label:6s}: [{bin_edges[i]:.1f}, {bin_edges[i+1]:.1f}]')

print('\nDistribución de clases:')
print(df[POP_CLASS_COL].value_counts(normalize=True).rename('proporción').to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df[POP_CLASS_COL].value_counts().reindex(CLASS_LABELS)
bars = ax.bar(counts.index.astype(str), counts.values, color=['#5B8DB8', '#E89B3C', '#27AE60'])
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v, f'{v:,}\n({v/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10)
ax.set_title('Target multiclase — clases balanceadas vía terciles', fontweight='bold')
ax.set_ylim(0, counts.max() * 1.18)
plt.tight_layout(); plt.show()

---
## 3. Split Estratificado (ANTES de cualquier encoding)

**Por qué este orden importa:** el `TargetMeanEncoder` (sección 4) usa la variable objetivo para construir un feature. Si lo aplicáramos al dataset completo y luego dividiéramos, los promedios por género estarían "contaminados" con información de las filas del test set → **data leakage** → métricas infladas en evaluación que no se sostienen en producción.

**Decisiones:**
- `test_size=0.20` — convención y suficiente con 110k+ filas.
- `stratify=y_class` — preserva proporción de las 3 clases en train y test. Crítico en clasificación.
- `random_state=42` — reproducibilidad.
- Conservamos `y_train_numeric` (popularity 0-100 original) **solo** para que el encoder pueda calcular medias por género sobre el train. **Nunca** se le pasa a un modelo.

In [ ]:
# Features = todo menos los dos targets (numérico y clase)
feature_cols = [c for c in df.columns if c not in (POP_NUMERIC_COL, POP_CLASS_COL)]
X = df[feature_cols].copy()
y_class = df[POP_CLASS_COL].copy()
y_numeric = df[POP_NUMERIC_COL].copy()    # solo para fit del target encoder

X_train, X_test, y_train, y_test, y_train_numeric, _ = train_test_split(
    X, y_class, y_numeric,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_class,
)

print(f'X_train: {X_train.shape}   y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}   y_test : {y_test.shape}')
print('\nDistribución de clases — train vs test (debe ser idéntica):')
comp = pd.DataFrame({
    'train_%': y_train.value_counts(normalize=True).mul(100).round(2),
    'test_%':  y_test.value_counts(normalize=True).mul(100).round(2),
})
print(comp)

---
## 4. Target Encoding sin Data Leakage

### Por qué un transformer custom y no `df.groupby` manual

Implementamos `TargetMeanEncoder` siguiendo la API de scikit-learn (`fit` / `transform`). Esto nos da:

- **Encapsulación del estado:** las medias por categoría quedan dentro del objeto, no en variables sueltas.
- **Reusabilidad:** se puede insertar en un `Pipeline` o `ColumnTransformer` cuando escalemos a producción.
- **Manejo explícito de categorías no vistas en test:** fallback a la media global del train (clave en producción real).
- **Smoothing opcional:** mezclamos la media por género con la media global, ponderando por tamaño de la categoría. Esto reduce overfitting en géneros con pocas observaciones.

**Suavizado bayesiano:** `encoded(g) = (n_g · mean_g + m · global_mean) / (n_g + m)`. Para géneros con `n_g >> m`, predomina la media local; para géneros raros, se acerca a la global.

In [ ]:
class TargetMeanEncoder(BaseEstimator, TransformerMixin):
    """Target mean encoder con smoothing bayesiano y fallback a media global.

    Reemplaza una columna categórica de alta cardinalidad por la media del
    target asociada a cada categoría, calculada exclusivamente sobre el set
    de entrenamiento (cero data leakage).

    Args:
        column: nombre de la columna categórica a codificar.
        smoothing: parámetro ``m`` del suavizado bayesiano. Valores más altos
            empujan las medias por categoría hacia la media global. ``0``
            desactiva el suavizado (no recomendado en producción).
        output_name: nombre de la nueva columna numérica generada.
    """

    def __init__(self, column: str, smoothing: float = 20.0,
                 output_name: str | None = None):
        self.column = column
        self.smoothing = smoothing
        self.output_name = output_name or f'{column}_te'

    def fit(self, X: pd.DataFrame, y: pd.Series) -> 'TargetMeanEncoder':
        if self.column not in X.columns:
            raise KeyError(f"Columna '{self.column}' no está en X.")
        if len(X) != len(y):
            raise ValueError('X e y deben tener la misma longitud.')

        y_arr = np.asarray(y, dtype=float)
        self.global_mean_ = float(y_arr.mean())

        agg = (
            pd.DataFrame({self.column: X[self.column].values, '_y': y_arr})
            .groupby(self.column)['_y']
            .agg(['mean', 'count'])
        )
        smoothed = (
            agg['count'] * agg['mean'] + self.smoothing * self.global_mean_
        ) / (agg['count'] + self.smoothing)
        self.mapping_ = smoothed.to_dict()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not hasattr(self, 'mapping_'):
            raise RuntimeError('Llama a fit() antes de transform().')
        X_out = X.copy()
        encoded = X_out[self.column].map(self.mapping_).fillna(self.global_mean_)
        X_out[self.output_name] = encoded.astype(float)
        X_out = X_out.drop(columns=[self.column])
        return X_out

    def fit_transform(self, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        return self.fit(X, y).transform(X)

In [ ]:
# Fit del encoder usando SOLO el train, sobre la popularity numérica original
encoder = TargetMeanEncoder(column='track_genre', smoothing=20.0,
                            output_name='genre_pop_te')
X_train_enc = encoder.fit_transform(X_train, y_train_numeric)
X_test_enc  = encoder.transform(X_test)

print('Cantidad de géneros aprendidos en train:', len(encoder.mapping_))
print(f'Media global aprendida del train: {encoder.global_mean_:.3f}')

# Sanity check: ¿hay géneros en test que no estaban en train? → fallback funciona
genres_train = set(X_train['track_genre'].unique())
genres_test  = set(X_test['track_genre'].unique())
unseen = genres_test - genres_train
print(f'Géneros en test no vistos en train: {len(unseen)} (manejados por fallback)')

print('\nMuestra del train ya codificado:')
X_train_enc.head(3)

---
## 5. Escalado de Features (StandardScaler)

- **Decision Tree / Random Forest:** son invariantes a la escala (deciden por umbrales). Para ellos usamos los datos **sin escalar** — preserva la interpretabilidad de los splits.
- **SVC:** depende de distancias en el espacio de features. Sin escalado, `duration_ms` (escala de cientos de miles) domina sobre `danceability` (rango 0-1). Es **obligatorio** escalar.

**Regla de oro:** `fit` solo en train, `transform` en train y test. Si calculáramos media/std del dataset completo, filtramos información del test.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_enc),
    columns=X_train_enc.columns,
    index=X_train_enc.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_enc),
    columns=X_test_enc.columns,
    index=X_test_enc.index,
)

print('✅ Escalado aplicado (solo se usará para SVC).')
print(f'Shape final del set de entrenamiento: {X_train_enc.shape}')

---
## 6. Entrenamiento y Evaluación de los 3 Clasificadores

Definimos una función de evaluación única que recibe modelo y datos, devuelve un dict con métricas. Esto evita repetir código y garantiza que los 3 modelos se midan con exactamente el mismo procedimiento.

**Métricas elegidas y por qué:**
- **Accuracy** — referencia rápida; útil porque las clases están balanceadas.
- **F1 macro** — promedia F1 de cada clase con peso igual. Penaliza si el modelo es bueno solo en clases mayoritarias.
- **F1 weighted** — pondera por soporte. Útil para reportar a stakeholders.
- **Matriz de confusión** — diagnóstico cualitativo: ¿qué clases se confunden entre sí?

In [ ]:
@dataclass
class ModelResult:
    """Contenedor inmutable con resultados de evaluación de un modelo."""
    name: str
    accuracy: float
    f1_macro: float
    f1_weighted: float
    confusion: np.ndarray
    labels: list = field(default_factory=list)
    report: str = ''


def evaluate_classifier(
    name: str,
    model,
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_te: pd.DataFrame, y_te: pd.Series,
    labels: Iterable[str] = CLASS_LABELS,
) -> ModelResult:
    """Entrena ``model`` y evalúa sobre el test set con métricas estándar de clasificación.

    Args:
        name: identificador legible del modelo.
        model: instancia sklearn-compatible no entrenada.
        X_tr, y_tr: datos de entrenamiento.
        X_te, y_te: datos de evaluación.
        labels: orden canónico de las clases para la matriz de confusión.

    Returns:
        ``ModelResult`` con accuracy, F1 macro/weighted, matriz de confusión y reporte.
    """
    labels = list(labels)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    return ModelResult(
        name=name,
        accuracy=accuracy_score(y_te, y_pred),
        f1_macro=f1_score(y_te, y_pred, average='macro'),
        f1_weighted=f1_score(y_te, y_pred, average='weighted'),
        confusion=confusion_matrix(y_te, y_pred, labels=labels),
        labels=labels,
        report=classification_report(y_te, y_pred, labels=labels, digits=3),
    )

### 6.1 Modelo 1 — Decision Tree Classifier

Baseline interpretable. `max_depth=12` evita el sobreajuste que se da al dejar crecer el árbol libremente (memoriza el train), pero deja suficiente capacidad para captar interacciones no triviales entre features de audio.

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
res_tree = evaluate_classifier(
    'Decision Tree', tree_model,
    X_train_enc, y_train, X_test_enc, y_test,
)
print(res_tree.report)

### 6.2 Modelo 2 — Random Forest Classifier

Ensemble de árboles entrenados con bagging + subsampling de features. Reduce la varianza del Decision Tree individual. `n_jobs=-1` paraleliza por árbol — escala bien en CPU multi-core.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
res_rf = evaluate_classifier(
    'Random Forest', rf_model,
    X_train_enc, y_train, X_test_enc, y_test,
)
print(res_rf.report)

### 6.3 Modelo 3 — SVC (kernel RBF)

**Decisión de ingeniería crítica:** la complejidad de entrenamiento de SVC con kernel RBF está entre O(n²) y O(n³). Con ~91k filas de train, el entrenamiento puede tardar **horas** en Colab gratuito y agotar la RAM. Esto no es una limitación teórica del modelo — es una **realidad de producción** que se discute en cualquier proyecto serio.

**Solución estándar:** entrenamos SVC sobre un **subsample estratificado** de 15k filas del train. La evaluación sigue siendo sobre el **test set completo** (~22k filas), lo que mantiene la comparación con DT y RF perfectamente justa — todos se miden sobre los mismos datos no vistos.

In [ ]:
SVC_SUBSAMPLE_SIZE = 15_000

X_tr_svc, _, y_tr_svc, _ = train_test_split(
    X_train_scaled, y_train,
    train_size=SVC_SUBSAMPLE_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE,
)
print(f'Subsample estratificado para SVC: {len(X_tr_svc):,} filas (de {len(X_train_scaled):,})')
print('Distribución de clases en el subsample:')
print(y_tr_svc.value_counts(normalize=True).mul(100).round(2).to_string())

svc_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    cache_size=500,
    random_state=RANDOM_STATE,
)
res_svc = evaluate_classifier(
    'SVC (RBF)', svc_model,
    X_tr_svc, y_tr_svc, X_test_scaled, y_test,
)
print(res_svc.report)

---
## 7. Comparación Final

### 7.1 Tabla comparativa de métricas

In [ ]:
results = [res_tree, res_rf, res_svc]

summary = pd.DataFrame([
    {
        'Modelo': r.name,
        'Accuracy': r.accuracy,
        'F1 Macro': r.f1_macro,
        'F1 Weighted': r.f1_weighted,
    }
    for r in results
]).set_index('Modelo')

summary_styled = (
    summary.style
    .format('{:.4f}')
    .background_gradient(cmap='Greens')
    .set_caption('Comparación de los 3 clasificadores sobre el test set')
)
summary_styled

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
summary.plot(kind='bar', ax=ax, color=['#5B8DB8', '#E89B3C', '#27AE60'], width=0.75)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Comparación de modelos — Test set', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
plt.tight_layout(); plt.show()

### 7.2 Matrices de confusión lado a lado

Lectura: cada fila es la clase real, cada columna la predicha. La diagonal son aciertos. Fuera de la diagonal vemos qué clases se confunden entre sí — típicamente `Media` se confunde con sus vecinas `Baja` y `Alta`, lo cual es esperable (es la clase "intermedia" sin frontera natural).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, r in zip(axes, results):
    sns.heatmap(
        r.confusion, annot=True, fmt='d', cmap='Blues',
        xticklabels=r.labels, yticklabels=r.labels,
        cbar=False, ax=ax, annot_kws={'size': 11},
    )
    ax.set_title(f'{r.name}\nAcc={r.accuracy:.3f}  F1m={r.f1_macro:.3f}', fontweight='bold')
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
plt.tight_layout(); plt.show()

### 7.3 Elección del modelo ganador

El criterio dominante es **F1 macro** porque trata las 3 clases con igual peso — coherente con nuestro diseño balanceado del target.

In [ ]:
winner = summary['F1 Macro'].idxmax()
winner_row = summary.loc[winner]
print(f'🏆 Modelo ganador (criterio: F1 Macro): {winner}')
print(f'   Accuracy    = {winner_row["Accuracy"]:.4f}')
print(f'   F1 Macro    = {winner_row["F1 Macro"]:.4f}')
print(f'   F1 Weighted = {winner_row["F1 Weighted"]:.4f}')

---
## 8. Conclusiones y Próximos Pasos

### Lo que se hizo

1. **Rediseño del problema** de regresión a clasificación multiclase con clases balanceadas por terciles dinámicos.
2. **Pipeline robusto sin data leakage**: split estratificado primero, encoder y scaler ajustados solo en train.
3. **Target encoding production-ready** con smoothing bayesiano y fallback para categorías no vistas, encapsulado en un transformer sklearn-compatible.
4. **Comparación honesta** de 3 familias de modelos con métricas idénticas sobre el mismo test set.
5. **Decisión de ingeniería documentada** para SVC: subsample estratificado en lugar de fingir que escala a 91k filas.

### Próximas iteraciones (si el proyecto continuara)

- **Cross-validation estratificada** (`StratifiedKFold`) para que las métricas tengan intervalos de confianza, no solo un punto.
- **Hyperparameter tuning** con `RandomizedSearchCV` sobre `max_depth`, `n_estimators`, `C`, `gamma`.
- **Encapsular todo en `Pipeline`** + persistir con `joblib` para deploy real.
- **Comparar contra un baseline trivial** (`DummyClassifier(strategy='stratified')`) para tener un piso absoluto.
- **Análisis de errores por clase** y por género para entender en qué segmentos del catálogo falla más el modelo.

---
*Entrega Final completada.*